# AgroPest-12 Part B: Stratified Augmentation-Group Split

This notebook creates the shared Kaggle dataset for Assignment B. It preserves the existing validation and test splits and divides the existing clean training split into disjoint SSL-only (80%) and labelled fine-tuning (20%) partitions.

The training files are treated as consecutive groups of three: the original image and its two augmented versions. A full group is assigned to one partition only. Groups are stratified by their dominant annotated class so both partitions preserve the original training class ratio.

In [1]:
# ============================================================
# CONFIG -- shared Part B dataset
# ============================================================
from pathlib import Path

CLEANED_PATH = Path("/kaggle/input/datasets/sadikshahriar/agropest-12-cleaned/AgroPest-12-Cleaned")
OUTPUT_ROOT = Path("/kaggle/working/AgroPest-12-PartB-Safe")

SEED = 445
AUGMENTATION_GROUP_SIZE = 3
SSL_FRACTION = 0.80
EXPECTED = {"train": 11475, "valid": 1089, "test": 543}
EXPECTED_SSL = 9180
EXPECTED_FINETUNE = 2295

CLASS_NAMES = [
    "Ants", "Bees", "Beetles", "Caterpillars",
    "Earthworms", "Earwigs", "Grasshoppers", "Moths",
    "Slugs", "Snails", "Wasps", "Weevils",
]

assert CLEANED_PATH.is_dir(), f"Cleaned dataset not found: {CLEANED_PATH}"
assert EXPECTED_SSL + EXPECTED_FINETUNE == EXPECTED["train"]
assert EXPECTED_SSL % AUGMENTATION_GROUP_SIZE == 0
assert EXPECTED_FINETUNE % AUGMENTATION_GROUP_SIZE == 0
print(f"CONFIG loaded | seed={SEED} | groups of {AUGMENTATION_GROUP_SIZE} | SSL/fine-tune={SSL_FRACTION:.0%}/{1-SSL_FRACTION:.0%}")

CONFIG loaded | seed=445 | groups of 3 | SSL/fine-tune=80%/20%


In [2]:
# ============================================================
# Imports and cleaned-dataset integrity audit
# ============================================================
import json
import shutil
from collections import Counter

import pandas as pd
import yaml
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

def image_files(directory):
    # Sorting makes the consecutive-triplet grouping deterministic and reproducible.
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)

def label_path(image_path):
    return image_path.parent.parent / "labels" / f"{image_path.stem}.txt"

source_images = {}
for split in ("train", "valid", "test"):
    image_dir = CLEANED_PATH / split / "images"
    label_dir = CLEANED_PATH / split / "labels"
    images = image_files(image_dir)
    image_stems = {path.stem for path in images}
    label_stems = {path.stem for path in label_dir.glob("*.txt")}
    assert len(images) == EXPECTED[split], f"Unexpected {split} count: {len(images)}"
    assert image_stems == label_stems, f"Image/label pairing error in {split}"
    source_images[split] = images
    print(f"{split:>5}: {len(images):>5} images | {len(label_stems):>5} labels | pairing verified")

assert len(source_images["train"]) % AUGMENTATION_GROUP_SIZE == 0
print(f"Training augmentation groups: {len(source_images['train']) // AUGMENTATION_GROUP_SIZE}")

train: 11475 images | 11475 labels | pairing verified
valid:  1089 images |  1089 labels | pairing verified
 test:   543 images |   543 labels | pairing verified
Training augmentation groups: 3825


## Consecutive Triplets And Stratification

Each sorted consecutive triplet is an indivisible augmentation family. Its dominant class is determined from the three corresponding YOLO label files. Stratified sampling operates on groups, not images, which prevents an original image and its augmented variants from crossing the 80/20 split.

In [3]:
# ============================================================
# Build augmentation families and make the 80/20 group split
# ============================================================
def classes_in_label(label_file):
    classes = []
    for line in label_file.read_text(encoding="utf-8").splitlines():
        fields = line.split()
        if len(fields) >= 5:
            classes.append(int(float(fields[0])))
    return classes

train_groups = []
train_images = source_images["train"]
for start in range(0, len(train_images), AUGMENTATION_GROUP_SIZE):
    members = train_images[start:start + AUGMENTATION_GROUP_SIZE]
    assert len(members) == AUGMENTATION_GROUP_SIZE
    class_counts = Counter(class_id for path in members for class_id in classes_in_label(label_path(path)))
    assert class_counts, f"No valid annotations in group starting with {members[0].name}"
    dominant_class = min(class_id for class_id, count in class_counts.items() if count == max(class_counts.values()))
    train_groups.append({
        "group_id": f"train_group_{start // AUGMENTATION_GROUP_SIZE:04d}",
        "members": members,
        "dominant_class_id": dominant_class,
        "dominant_class": CLASS_NAMES[dominant_class],
    })

# Inspect these names once on Kaggle to confirm the sorted triplets match the known augmentation order.
display(pd.DataFrame([{
    "group_id": group["group_id"], "dominant_class": group["dominant_class"],
    "files": " | ".join(path.name for path in group["members"]),
} for group in train_groups[:10]]))

group_labels = [group["dominant_class_id"] for group in train_groups]
splitter = StratifiedShuffleSplit(n_splits=1, train_size=SSL_FRACTION, random_state=SEED)
ssl_indices, finetune_indices = next(splitter.split(train_groups, group_labels))
ssl_indices, finetune_indices = set(ssl_indices), set(finetune_indices)

for index, group in enumerate(train_groups):
    group["partition"] = "ssl_pool" if index in ssl_indices else "finetune_20"

assert sum(len(group["members"]) for group in train_groups if group["partition"] == "ssl_pool") == EXPECTED_SSL
assert sum(len(group["members"]) for group in train_groups if group["partition"] == "finetune_20") == EXPECTED_FINETUNE

group_summary = pd.DataFrame([{
    "partition": group["partition"], "dominant_class": group["dominant_class"]
} for group in train_groups])
display(pd.crosstab(group_summary["dominant_class"], group_summary["partition"], margins=True))

,group_id,dominant_class,files
0,train_group_0000,Wasps,WASP-1-_jpg.rf.2d2f246bcb28271f492b65ae2241707...
1,train_group_0001,Wasps,WASP-1-_png.rf.a3024df45b96aaf6cf640799fc3ea2b...
2,train_group_0002,Weevils,Weevil-10-_jpg.rf.0b7a0d5d9f9f5ee74f45ffeca1bd...
3,train_group_0003,Weevils,Weevil-103-_jpg.rf.d2e004cb51f548178a08bf38ea9...
4,train_group_0004,Weevils,Weevil-104-_jpg.rf.0773d4f6a9b020ea717639e8a0e...
5,train_group_0005,Weevils,Weevil-106-_jpg.rf.03b49c9378a72dc1d2a1fbd0d9b...
6,train_group_0006,Weevils,Weevil-107-_jpg.rf.5882ee7c5b8d25d73c6131c2d93...
7,train_group_0007,Weevils,Weevil-11-_jpg.rf.6a3a9bc19737d151fdbc34259446...
8,train_group_0008,Weevils,Weevil-110-_jpg.rf.5aa517bf763a54dcdbf24a1bcdf...
9,train_group_0009,Weevils,Weevil-113-_jpg.rf.9ca64c56b93a6dd0fcfc61fb442...


partition,finetune_20,ssl_pool,All
dominant_class,,,
Ants,69,275,344
Bees,72,290,362
Beetles,57,228,285
Caterpillars,61,242,303
Earthworms,47,190,237
Earwigs,63,251,314
Grasshoppers,70,278,348
Moths,70,282,352
Slugs,53,213,266


In [4]:
# ============================================================
# Build manifest and materialize the reusable output dataset
# ============================================================
records = []
for group in train_groups:
    for path in group["members"]:
        records.append({
            "filename": path.name, "source_split": "train",
            "partition": group["partition"], "group_id": group["group_id"],
            "dominant_group_class": group["dominant_class"], "source_path": str(path),
        })
for split in ("valid", "test"):
    for index, path in enumerate(source_images[split]):
        records.append({
            "filename": path.name, "source_split": split, "partition": split,
            "group_id": f"{split}_preserved_{index:04d}", "dominant_group_class": None,
            "source_path": str(path),
        })
manifest = pd.DataFrame(records)
expected_counts = {"ssl_pool": EXPECTED_SSL, "finetune_20": EXPECTED_FINETUNE, "valid": EXPECTED["valid"], "test": EXPECTED["test"]}
assert manifest["partition"].value_counts().to_dict() == expected_counts
assert manifest.groupby("group_id")["partition"].nunique().max() == 1

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

output_layout = {
    "ssl_pool": (OUTPUT_ROOT / "ssl_pool" / "images", None),
    "finetune_20": (OUTPUT_ROOT / "finetune_20" / "images", OUTPUT_ROOT / "finetune_20" / "labels"),
    "valid": (OUTPUT_ROOT / "valid" / "images", OUTPUT_ROOT / "valid" / "labels"),
    "test": (OUTPUT_ROOT / "test" / "images", OUTPUT_ROOT / "test" / "labels"),
}
for image_dir, labels_dir in output_layout.values():
    image_dir.mkdir(parents=True, exist_ok=True)
    if labels_dir is not None:
        labels_dir.mkdir(parents=True, exist_ok=True)

for record in tqdm(records, desc="Copying Part B dataset"):
    source_image = Path(record["source_path"])
    image_dir, labels_dir = output_layout[record["partition"]]
    shutil.copy2(source_image, image_dir / source_image.name)
    if labels_dir is not None:
        source_label = label_path(source_image)
        assert source_label.is_file(), f"Missing label: {source_label}"
        shutil.copy2(source_label, labels_dir / source_label.name)

data_config = {
    "path": ".", "train": "finetune_20/images", "val": "valid/images",
    "test": "test/images", "nc": len(CLASS_NAMES), "names": CLASS_NAMES,
}
with (OUTPUT_ROOT / "data_20.yaml").open("w", encoding="utf-8") as handle:
    yaml.safe_dump(data_config, handle, sort_keys=False)
manifest.drop(columns="source_path").to_csv(OUTPUT_ROOT / "partition_manifest.csv", index=False)

Copying Part B dataset:   0%|          | 0/13107 [00:00<?, ?it/s]

In [5]:
# ============================================================
# Verify output and report class-ratio preservation
# ============================================================
def instance_counts(labels_dir):
    counts = Counter()
    for label_file in labels_dir.glob("*.txt"):
        counts.update(classes_in_label(label_file))
    return [counts[class_id] for class_id in range(len(CLASS_NAMES))]

for partition, (image_dir, labels_dir) in output_layout.items():
    image_count = len(image_files(image_dir))
    assert image_count == expected_counts[partition], f"Unexpected {partition} output count"
    if labels_dir is None:
        assert not any((OUTPUT_ROOT / "ssl_pool").rglob("*.txt")), "SSL pool must not contain labels"
    else:
        assert len(list(labels_dir.glob("*.txt"))) == image_count

train_instances = instance_counts(CLEANED_PATH / "train" / "labels")
# SSL labels are deliberately absent from the output. Compute its class counts from source labels via the manifest.
ssl_counts = Counter()
for record in records:
    if record["partition"] == "ssl_pool":
        ssl_counts.update(classes_in_label(label_path(Path(record["source_path"]))))
finetune_instances = instance_counts(OUTPUT_ROOT / "finetune_20" / "labels")

class_report = pd.DataFrame({
    "class": CLASS_NAMES,
    "original_train_instances": train_instances,
    "ssl_instances": [ssl_counts[index] for index in range(len(CLASS_NAMES))],
    "finetune_instances": finetune_instances,
})
class_report["original_train_pct"] = 100 * class_report["original_train_instances"] / class_report["original_train_instances"].sum()
class_report["ssl_pct"] = 100 * class_report["ssl_instances"] / class_report["ssl_instances"].sum()
class_report["finetune_pct"] = 100 * class_report["finetune_instances"] / class_report["finetune_instances"].sum()
class_report["finetune_minus_original_pp"] = class_report["finetune_pct"] - class_report["original_train_pct"]
class_report.to_csv(OUTPUT_ROOT / "class_distribution_report.csv", index=False)
display(class_report.round(2))

report = {
    "protocol": "Existing validation/test preserved; sorted consecutive training triplets treated as augmentation families; stratified disjoint 80/20 group split.",
    "seed": SEED, "augmentation_group_size": AUGMENTATION_GROUP_SIZE,
    "image_counts": expected_counts, "source_counts": EXPECTED,
    "class_distribution_file": "class_distribution_report.csv",
}
with (OUTPUT_ROOT / "partition_report.json").open("w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)

readme = """# AgroPest-12 Part B Safe Splits

The original validation and test splits are preserved. The original training images are grouped in sorted consecutive triplets, then stratified by dominant group class into disjoint 80% SSL-only and 20% labelled fine-tuning partitions.

- `ssl_pool/images`: 9,180 images; no labels are included.
- `finetune_20`: 2,295 labelled images.
- `valid` and `test`: unchanged original clean splits.

Every downstream notebook must use this same `finetune_20`, validation, and test data.
"""
(OUTPUT_ROOT / "README.md").write_text(readme, encoding="utf-8")
print(f"Success. Publish this directory as a Kaggle dataset:\n{OUTPUT_ROOT}")

,class,original_train_instances,ssl_instances,finetune_instances,original_train_pct,ssl_pct,finetune_pct,finetune_minus_original_pp
0,Ants,2228,1804,424,14.61,14.80,13.88,-0.73
1,Bees,1575,1215,360,10.33,9.97,11.79,1.46
2,Beetles,1055,800,255,6.92,6.56,8.35,1.43
3,Caterpillars,1740,1410,330,11.41,11.56,10.81,-0.61
4,Earthworms,1083,894,189,7.10,7.33,6.19,-0.91
5,Earwigs,1182,956,226,7.75,7.84,7.40,-0.35
6,Grasshoppers,1071,859,212,7.02,7.05,6.94,-0.08
7,Moths,1059,849,210,6.95,6.96,6.88,-0.07
8,Slugs,918,742,176,6.02,6.09,5.76,-0.26
9,Snails,1193,977,216,7.83,8.01,7.07,-0.75


Success. Publish this directory as a Kaggle dataset:
/kaggle/working/AgroPest-12-PartB-Safe


## Kaggle Hand-off

Run all cells, save a successful Kaggle version, and publish `/kaggle/working/AgroPest-12-PartB-Safe/` as a dataset. Attach it to every Part B notebook. SSL notebooks use only `ssl_pool/images`; downstream notebooks use `finetune_20`, `valid`, and `test`.